# Lightweight IDS Using Machine Learning for IoT Devices
## Notebook 05: Data Leakage Check
**Author:** Mimi Nguyet Mi Taylor  
**Programme:** MSc Computer Science, Birkbeck, University of London  
**Supervisor:** Professor Paul Yoo
---
### About this notebook

This notebook runs two checks that support statements in the report. Neither is part of the main pipeline.

Check 1: feature selection and the train/test split. Feature selection in this project was fitted on all cleaned records before the train/test split, as described in section 6.6 of the report. This notebook refits the same selection step on the training partition only and compares the resulting feature set against the one used.

Check 2: correlation among the selected features. Section 6.3 attributes Naive Bayes' weak performance to its independence assumption. This notebook reports the correlation matrix for the 13 selected features, showing where that assumption does not hold.

The cleaning cells are copied unchanged from Notebook 02:

Load the stratified sample (stratified_sample_1m.csv)
Data cleaning: missing values, duplicates, the raw Label column and infinite values
Reproduce the original feature selection, then refit the same step on the training partition only and compare
Compute the correlation matrix for the 13 selected features
   
This notebook writes no files and does not affect the outputs of Notebooks 01 to 04.

### Input
stratified_sample_1m.csv (produced by Notebook 01)
### Output
None. Results are printed below.

In [1]:
# Import all libraries needed for this notebook.
# pandas/numpy: data loading and manipulation
# matplotlib/seaborn: plotting
# joblib: saving data to disk
# sklearn: feature selection, train/test split, Random Forest
# imblearn: SMOTE-ENN for class imbalance correction

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from imblearn.combine import SMOTEENN

RANDOM_STATE = 42

print("All imports successful.")

All imports successful.


In [2]:
# Load the stratified sample created in Notebook 01.
# DATA_PATH is the full file path to the saved CSV file.
# This is the starting point for all cleaning and preprocessing steps.

DATA_PATH = os.path.expanduser("~/IDS_PROJECT_REPORT/stratified_sample_1m.csv")

df = pd.read_csv(DATA_PATH)

# Capture the original load count here, before cleaning steps below
# reuse and overwrite the df variable to keep record of what was actually loaded, for the notebook summary.
initial_rows = len(df)
initial_cols = df.shape[1]

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns.")
print(f"\nClass distribution (before cleaning):")
print(df['Category'].value_counts())

Loaded 1,000,000 rows and 41 columns.

Class distribution (before cleaning):
Category
DDoS              722718
DoS               172072
Mirai              56014
Benign             23354
Reconnaissance     14685
Spoofing           10350
Web-based            529
Brute Force          278
Name: count, dtype: int64


In [3]:
# Step 1: Check for missing values.
# Missing values can cause classifiers to crash or produce wrong results.

missing = df.isnull().sum()
missing_cols = missing[missing > 0]

print(f"Columns with missing values: {len(missing_cols)}")
if len(missing_cols) > 0:
    print(missing_cols)

rows_before = len(df)
df = df.dropna()
rows_after = len(df)

print(f"\nRows removed due to missing values: {rows_before - rows_after:,}")
print(f"Rows remaining: {rows_after:,}")

Columns with missing values: 2
Std         14
Variance    14
dtype: int64

Rows removed due to missing values: 14
Rows remaining: 999,986


In [4]:
# Step 2: Remove duplicate rows.
# Duplicates can skew results by making the classifier see the same 
# record multiple times, artificially inflating performance.

rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)

print(f"Duplicate rows removed: {rows_before - rows_after:,}")
print(f"Rows remaining: {rows_after:,}")

Duplicate rows removed: 268,771
Rows remaining: 731,215


In [5]:
# Step 3: Inspect the column list before dropping the raw Label column.
# The 39 features are all flow-level, so there are no identifier
# columns to remove.

print("All columns:")
print(df.columns.tolist())

All columns:
['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance', 'Label', 'Category']


In [6]:
# Remove the Label column. This was the original raw attack label 
# (e.g. DDOS-ICMP_FLOOD) which we mapped to Category in Notebook 01.
# Label must be removed because it would give the classifier a direct
# clue about the answer, making results meaningless.
# Category is our target variable and is kept separately below.
# Note: total columns drops from 41 to 40 here, since Label is removed.

df = df.drop(columns=['Label'])

print(f"Columns remaining: {df.shape[1]}")
print(f"Columns: {df.columns.tolist()}")

Columns remaining: 40
Columns: ['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance', 'Category']


In [7]:
# Separate features (X) and target label (y).
# X contains everything we use to make predictions (the 39 feature columns).
# y contains what we are trying to predict (the attack category).
# We also replace any infinite values with NaN and drop those rows.
# Infinite values appear in some network flow features due to division 
# by zero in flow rate calculations and will crash most classifiers.
# Note: total columns drops from 40 to 39 here, since Category is separated out into y.

X = df.drop(columns=['Category'])
y = df['Category']

# Check for infinite values
inf_count = np.isinf(X.select_dtypes(include=[np.number])).sum().sum()
print(f"Infinite values found: {inf_count}")

X = X.replace([np.inf, -np.inf], np.nan)
mask = X.notna().all(axis=1)
X = X[mask]
y = y[mask]

print(f"Rows remaining after removing infinite values: {len(X):,}")
print(f"\nFinal class distribution after cleaning:")
print(y.value_counts())

Infinite values found: 10
Rows remaining after removing infinite values: 731,205

Final class distribution after cleaning:
Category
DDoS              487346
DoS               139145
Mirai              55876
Benign             23342
Reconnaissance     14675
Spoofing           10014
Web-based            529
Brute Force          278
Name: count, dtype: int64


In [8]:
# Feature selection using Random Forest importance scores.
# We train a small Random Forest purely to score how useful each 
# feature is for predicting attack category.
# Features with very low importance scores add noise and slow down 
# all classifiers without improving accuracy.
# Random Forest is used here because Phan et al. (2024) found it 
# the most stable feature selection method across all classifiers 
# on CICIoT2023.

print("Training Random Forest for feature importance scoring...")
print("This may take a few minutes.")

X_numeric = X.select_dtypes(include=[np.number])

rf_selector = RandomForestClassifier(
    n_estimators=50,    # 50 trees is enough for importance scoring
    max_depth=10,       # shallow trees are faster
    random_state=RANDOM_STATE,
    n_jobs=-1           # use all CPU cores
)
rf_selector.fit(X_numeric, y)

importances = pd.Series(rf_selector.feature_importances_, index=X_numeric.columns)
importances = importances.sort_values(ascending=False)

print(f"\nTop 20 features by importance:")
print(importances.head(20))

Training Random Forest for feature importance scoring...
This may take a few minutes.

Top 20 features by importance:
Tot sum            0.117622
Tot size           0.113498
AVG                0.110445
Protocol Type      0.065786
Max                0.059502
Std                0.052662
Number             0.046959
Header_Length      0.043671
Rate               0.042872
Min                0.039070
ICMP               0.029800
ack_flag_number    0.028626
Variance           0.026528
TCP                0.023966
psh_flag_number    0.023788
ack_count          0.023536
Time_To_Live       0.020903
HTTPS              0.020074
IAT                0.019504
UDP                0.018285
dtype: float64


In [9]:
# Refits feature selection on the training partition only and compares it
# against the feature set used in this project.

from sklearn.model_selection import train_test_split as _tts
from sklearn.ensemble import RandomForestClassifier as _RF
import pandas as _pd

# The selection actually used: fitted on all cleaned records (cell above)
original = sorted(importances[importances >= 0.025].index.tolist())

# The corrected design: same scorer, fitted on the training partition only
X_tr_check, X_te_check, y_tr_check, y_te_check = _tts(
    X_numeric, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
print(f"Training partition: {len(X_tr_check):,} rows, {X_tr_check.shape[1]} features")

rf_check = _RF(n_estimators=50, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
rf_check.fit(X_tr_check, y_tr_check)

imp_check = _pd.Series(
    rf_check.feature_importances_, index=X_tr_check.columns
).sort_values(ascending=False)
selected_check = sorted(imp_check[imp_check >= 0.025].index.tolist())

print(f"\nOriginal selection (all {len(X_numeric):,} records): {len(original)} features")
print(original)
print(f"\nTraining partition only ({len(X_tr_check):,} records): {len(selected_check)} features")
print(selected_check)
print(f"\nIdentical: {set(original) == set(selected_check)}")
if set(original) != set(selected_check):
    print("  Only in original:", sorted(set(original) - set(selected_check)))
    print("  Only in check   :", sorted(set(selected_check) - set(original)))

print("\nImportances near the 0.025 threshold:")
print(imp_check[(imp_check > 0.015) & (imp_check < 0.045)])

Training partition: 511,843 rows, 39 features

Original selection (all 731,205 records): 13 features
['AVG', 'Header_Length', 'ICMP', 'Max', 'Min', 'Number', 'Protocol Type', 'Rate', 'Std', 'Tot size', 'Tot sum', 'Variance', 'ack_flag_number']

Training partition only (511,843 records): 14 features
['AVG', 'HTTPS', 'Header_Length', 'ICMP', 'Max', 'Min', 'Number', 'Protocol Type', 'Rate', 'Std', 'Tot size', 'Tot sum', 'UDP', 'Variance']

Identical: False
  Only in original: ['ack_flag_number']
  Only in check   : ['HTTPS', 'UDP']

Importances near the 0.025 threshold:
Header_Length      0.040432
Min                0.040191
Variance           0.038241
Std                0.031302
ICMP               0.031031
HTTPS              0.027640
UDP                0.025857
psh_flag_number    0.022785
Time_To_Live       0.021743
TCP                0.021553
ack_count          0.021473
IAT                0.021270
ack_flag_number    0.020002
dtype: float64


In [10]:
# Correlation among the 13 selected features, supporting the
# independence-assumption discussion in section 6.3 of the report.
selected = ['Tot sum','Tot size','AVG','Protocol Type','Max','Std','Number',
            'Header_Length','Rate','Min','ICMP','ack_flag_number','Variance']
print(X_numeric[selected].corr().round(2))

                 Tot sum  Tot size   AVG  Protocol Type   Max   Std  Number  \
Tot sum             1.00      0.73  0.73           0.51  0.43  0.39    0.11   
Tot size            0.73      1.00  1.00           0.34  0.77  0.76   -0.34   
AVG                 0.73      1.00  1.00           0.34  0.77  0.76   -0.34   
Protocol Type       0.51      0.34  0.34           1.00  0.10 -0.03    0.06   
Max                 0.43      0.77  0.77           0.10  1.00  0.93   -0.39   
Std                 0.39      0.76  0.76          -0.03  0.93  1.00   -0.46   
Number              0.11     -0.34 -0.34           0.06 -0.39 -0.46    1.00   
Header_Length      -0.38     -0.09 -0.09          -0.39  0.09  0.14   -0.34   
Rate               -0.21     -0.22 -0.22          -0.15 -0.18 -0.16    0.11   
Min                 0.42      0.51  0.51           0.40  0.23  0.07   -0.09   
ICMP                0.02     -0.03 -0.03          -0.34 -0.03  0.01    0.10   
ack_flag_number    -0.07      0.19  0.19          -0